In [3]:
import requests
import pandas as pd
from io import StringIO

# Centroid of DK1 (Jutland)
lat, lon = 56.2, 9.5

def fetch_open_meteo_forecasts():
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-03-01",
        "end_date": "2026-03-01", # Note: Future dates will return empty rows until they happen
        "models": "icon_seamless", # THE FIX: Deep historical operational archive
        "hourly": [
            "temperature_2m", 
            "relative_humidity_2m", 
            "cloud_cover", 
            "surface_pressure", 
            "wind_speed_10m", 
            "shortwave_radiation" 
        ],
        "timezone": "Europe/Copenhagen",
        "format": "csv"
    }
    
    print("Requesting 2 years of stitched ICON operational forecasts...")
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text), skiprows=3)
        
        # Sanity check to ensure we aren't getting empty data again
        if df['temperature_2m (°C)'].isnull().all():
            print("❌ Failure: The API returned valid headers but empty data columns.")
        else:
            df.to_csv("dk1_icon_historical_forecasts.csv", index=False)
            print("✅ Success. Data is populated and saved to dk1_icon_historical_forecasts.csv")
            # Show the first few rows of actual data
            print("\nPreview of the extracted data:")
            print(df.dropna().head())
            
    else:
        print(f"Error {response.status_code}: {response.text}")

fetch_open_meteo_forecasts()

Requesting 2 years of stitched ICON operational forecasts...
✅ Success. Data is populated and saved to dk1_icon_historical_forecasts.csv

Preview of the extracted data:
               time  temperature_2m (°C)  relative_humidity_2m (%)  \
0  2024-03-01T00:00                  6.2                        94   
1  2024-03-01T01:00                  6.0                        91   
2  2024-03-01T02:00                  5.9                        90   
3  2024-03-01T03:00                  5.6                        90   
4  2024-03-01T04:00                  4.9                        90   

   cloud_cover (%)  surface_pressure (hPa)  wind_speed_10m (km/h)  \
0              100                   997.3                   12.1   
1              100                   996.6                   11.8   
2               97                   996.3                   13.6   
3               94                   995.4                   14.6   
4               96                   995.3                   15.2

In [ ]:
import requests
import pandas as pd
from io import StringIO

# Cities and populations
cities = {
    "Aarhus":  {"lat": 56.1567, "lon": 10.2108, "pop": 290000},
    "Odense":  {"lat": 55.3959, "lon": 10.3883, "pop": 181000},
    "Aalborg": {"lat": 57.0488, "lon": 9.9217, "pop": 120000},
    "Esbjerg": {"lat": 55.4703, "lon": 8.4519,  "pop": 72000},
    "Randers": {"lat": 56.4606, "lon": 10.0364, "pop": 64000}
}

total_population = sum(city['pop'] for city in cities.values())

def fetch_city_data(lat, lon, city_name):
    """Fetches data for a single point from Open-Meteo."""
    url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-03-01",
        "end_date": "2026-03-01", 
        "models": "icon_seamless", 
        "hourly": ["temperature_2m", "relative_humidity_2m", "cloud_cover", 
                   "surface_pressure", "wind_speed_10m", "shortwave_radiation"],
        "timezone": "Europe/Copenhagen",
        "format": "csv"
    }
    
    print(f"Fetching data for {city_name}...")
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text), skiprows=3)
        df['time'] = pd.to_datetime(df['time'])
        df.set_index('time', inplace=True)
        return df
    else:
        print(f"Failed to fetch {city_name}. Code {response.status_code}")
        return None

# Main
all_weighted_data = []

for name, info in cities.items():
    df_city = fetch_city_data(info['lat'], info['lon'], name)
    
    if df_city is not None:
        # weights for each city
        weight = info['pop'] / total_population
        print(f"  Applying weight: {weight:.3f}")
        
        # weighting
        df_weighted = df_city * weight
        all_weighted_data.append(df_weighted)

# Aggregate the Data
if all_weighted_data:
    print("\nAggregating population-weighted proxy for DK1...")
    # final weighted average
    dk1_proxy = sum(all_weighted_data)
    
    # Save the final dataset
    dk1_proxy.to_csv("dk1_population_weighted_weather.csv")
    print("✅ Success! Dataset saved to dk1_population_weighted_weather.csv")
    
    print(dk1_proxy.dropna().head())
else:
    print("Failed to build dataset.")

Fetching data for Aarhus...
  Applying weight: 0.399
Fetching data for Odense...
  Applying weight: 0.249
Fetching data for Aalborg...
  Applying weight: 0.165
Fetching data for Esbjerg...
  Applying weight: 0.099
Fetching data for Randers...
  Applying weight: 0.088

Aggregating population-weighted proxy for DK1...
✅ Success! Dataset saved to dk1_population_weighted_weather.csv

Preview of DK1 Weighted Data:
                     temperature_2m (°C)  relative_humidity_2m (%)  \
time                                                                 
2024-03-01 00:00:00             6.302476                 84.869326   
2024-03-01 01:00:00             5.946492                 85.394773   
2024-03-01 02:00:00             5.728611                 86.675378   
2024-03-01 03:00:00             5.199450                 88.427785   
2024-03-01 04:00:00             4.807840                 89.573590   

                     cloud_cover (%)  surface_pressure (hPa)  \
time                            